# 构建 Makemore - 练习

来自[构建 makemore 视频](https://www.youtube.com/watch?v=PaCmpygFfXo)的练习。<br>
视频描述中包含了这些练习，下面也列出了它们。

1. 在 YouTube 上观看[构建 makemore 视频](https://www.youtube.com/watch?v=PaCmpygFfXo)
2. 回来完成练习来提升自己 :)

<style>
/* Keep notebook content printable without horizontal clipping. */
.jp-OutputArea-output img,
.jp-RenderedImage img,
img {
  max-width: 100% !important;
  height: auto !important;
}

.jp-Cell,
.jp-InputArea,
.jp-OutputArea-output,
.jp-RenderedMarkdown,
.text_cell_render,
.rendered_html {
  overflow-wrap: anywhere !important;
  word-break: break-word !important;
}

.jp-InputArea-editor pre,
.jp-RenderedText pre,
.jp-OutputArea-output pre,
.jp-OutputArea-output code,
.highlight pre,
.input_area pre,
.output_area pre,
.output_subarea pre,
.output_text pre,
.output_stream pre,
pre,
code {
  white-space: pre-wrap !important;
  overflow-wrap: anywhere !important;
  word-break: break-word !important;
}

.rendered_html table,
.jp-RenderedHTMLCommon table {
  max-width: 100% !important;
}
</style>


In [31]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm import tqdm
%matplotlib inline

## 练习 1 - 三元语言模型

**目标：**训练一个三元语言模型 (trigram language model)，即以两个字符作为输入来预测第 3 个字符。<br>
你可以随意使用计数法或神经网络。评估损失；它比 bigram 模型有所改善吗？

In [ ]:
# Set training device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
# Load dataset -> List[str]
words = open('../names.txt', 'r').read().splitlines()
g = torch.Generator(device=device).manual_seed(2147483647)

chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0 # Special token has position zero
itos = {i:s for s,i in stoi.items()}

# TODO: Modify this to accommodate for trigrams
for w in words[:1]:
    chs = ['.'] + list(w) + ['.']
    # Two char 'sliding-window'
    for ch1, ch2 in zip(chs, chs[1:]):
        print(ch1, ch2)

# -----
# TODO: Your code here
# Implement a trigram model
# -----

## 练习 2 - 划分数据集，在 dev 和 test 集上评估

**目标：**将数据集随机划分为 $80\%$ 的 `train` 集、$10\%$ 的 `dev` 集、$10\%$ 的 `test` 集。<br>
仅在 `training` 集上训练 bigram 和 trigram 模型。在 `dev` 和 `test` 划分上评估它们。

你能观察到什么？

In [5]:
g = torch.Generator(device=device).manual_seed(2147483647)

### 用 bigram 模型建立基线

我们使用视频中构建的 bigram 模型代码来建立一个基线。

In [ ]:
# Create set of all *bigrams*
xs, ys = [], []

for w in words:
    chs = ['.'] + list(w) + ['.']
    # Two char 'sliding-window'
    for ch1, ch2 in zip(chs, chs[1:]):
        xs.append(stoi[ch1])
        ys.append(stoi[ch2])

xs, ys = torch.tensor(xs), torch.tensor(ys) # [196113], [196113]
num_x, num_y = xs.nelement(), ys.nelement()

# TODO: Shuffle/Permute the dataset, keeping pairs in sync
# TODO: Split the dataset into 80:10:10 for train:valid:test
xs_bi_train, xs_bi_valid, xs_bi_test = None, None, None
ys_bi_train, ys_bi_valid, ys_bi_test = None, None, None

In [ ]:
W = torch.randn((27,27), device=device, generator=g, requires_grad=True)

# Training cycles, using the entire dataset -> 200 Epochs
for k in range(200):    
    # Forward pass
    xenc = F.one_hot(xs_bi_train, num_classes=27).float().to(device) # one-hot encode the names
    logits = xenc @ W # logits, different word for log-counts
    counts = logits.exp() # 'fake counts', kinda like in  the N matrix of bigram
    probs = counts / counts.sum(1, keepdims=True) # Normal distribution probabilities (this is y_pred)
    loss = -probs[torch.arange(len(probs)), ys_bi_train].log().mean() + 0.01 * (W**2).mean()
    print(f'Loss @ iteration {k+1}: {loss}')
    # Backward pass
    W.grad = None # Make sure all gradients are reset
    loss.backward() # Torch kept track of what this variable is, kinda cool
    # Weight update
    W.data += -50 * W.grad

In [ ]:
# Validation Loss
with torch.no_grad():
    xenc = F.one_hot(xs_bi_valid, num_classes=27).float().to(device) # one-hot encode the names
    logits = xenc @ W # logits, different word for log-counts
    counts = logits.exp() # 'fake counts', kinda like in  the N matrix of bigram
    probs = counts / counts.sum(1, keepdims=True) # Normal distribution probabilities (this is y_pred)
    loss = -probs[torch.arange(len(probs)), ys_bi_valid].log().mean() + 0.01 * (W**2).mean()
print(f'Validation Loss: {loss}')

# Test Loss
with torch.no_grad():
    xenc = F.one_hot(xs_bi_test, num_classes=27).float().to(device) # one-hot encode the names
    logits = xenc @ W # logits, different word for log-counts
    counts = logits.exp() # 'fake counts', kinda like in  the N matrix of bigram
    probs = counts / counts.sum(1, keepdims=True) # Normal distribution probabilities (this is y_pred)
    loss = -probs[torch.arange(len(probs)), ys_bi_test].log().mean() + 0.01 * (W**2).mean()
print(f'Test Loss:\t {loss}')

### 比较 bigram 和 trigram 模型

In [1]:
# TODO: Create set of all *trigrams*
xs, ys = [], []

# TODO: Shuffle/Permute the dataset, keeping (x,y) pairs in sync
# TODO: Split the dataset into 80:10:10 for train:valid:test
xs_tri_train, xs_tri_valid, xs_tri_test = None, None, None
ys_tri_train, ys_tri_valid, ys_tri_test = None, None, None

In [ ]:
# TODO: Implement and train a trigram model

In [ ]:
# TODO: Evaluate the trigram model on the validation and test sets

## 练习 3 - 调节平滑强度

**目标：**使用 *dev 集*来调节 trigram 模型的平滑（或正则化）强度——即<br>
尝试多种可能性，看看哪一个基于 dev 集损失表现最好。<br>
在调节这个强度时，你能在 train 和 dev 集损失中观察到什么规律？<br>
取最佳的平滑设置，在最后于 test 集上评估一次。<br>
你最终能达到多好的损失？

In [56]:
# TODO: Create set of all *trigrams*
xs, ys = [], []

# TODO: Shuffle/Permute the dataset, keeping (x,y) pairs in sync
# TODO: Split the dataset into 80:10:10 for train:valid:test
xs_tri_train, xs_tri_valid, xs_tri_test = None, None, None
ys_tri_train, ys_tri_valid, ys_tri_test = None, None, None

In [ ]:
# TODO: Build the hyperparameter search for regularization strength of the trigram model

## 练习 4 - 独热向量删除

**目标：**我们看到独热向量 (one-hot vectors) 只是选取了 $W$ 的一行，因此显式地生成这些向量显得很浪费。<br>
你能删除我们对 `F.one_hot` 的使用，改为直接索引 $W$ 的行吗？

In [ ]:
# TODO: Rewrite the training loop to delete F.one_hot

## 练习 5：使用 F.cross_entropy

**目标：**查找并改用 `F.cross_entropy`。你应该得到相同的结果。你能想到为什么我们更倾向于使用 `F.cross_entropy` 吗？这里是 [`F.cross_entropy` 的文档](https://pytorch.org/docs/stable/generated/torch.nn.functional.cross_entropy.html)。

In [ ]:
# TODO: Rewrite the training loop from Ex. 4 to employ F.cross_entropy

## 练习 6：元练习

**目标：**想一个有趣/有意思的练习并完成它。

In [ ]:
# TODO: The stage is yours!